# Notebook 087: Unity Catalog Volumes for Unstructured and Semi-Structured Data Governance

**Module:** Module 09: Databricks Platform, Unity Catalog, Security & FinOps  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 09: DBUtils & Video 14: Master Unity Catalog in Databricks  

---

## 1. Business Scenario & Real-World Context
A machine learning team manages non-tabular files (PDF documents, medical images, audio files, and model weights) in cloud storage. Legacy DBFS mounts lack row/file-level access control. The platform requires Unity Catalog Volumes for secure governance.

## 2. Technical Requirement & Problem Statement
Master Unity Catalog Volumes: compare Managed Volumes vs External Volumes, manage files inside volumes using `dbutils.fs` and standard POSIX `/Volumes/catalog/schema/volume` paths, apply governance privileges, and ingest raw documents into Spark.

## 3. Spark & Databricks Architecture Notes
Unity Catalog Volumes govern non-tabular files (CSVs, images, PDFs, models) alongside tabular tables. Unlike legacy DBFS mounts that grant broad workspace-wide access, Volumes enforce fine-grained access control using the standard 3-tier namespace (`/Volumes/catalog/schema/volume_name/`).


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_087_Unity_Catalog_Volumes_for_Unstructured_and_Semi-Structured_Data_Governance")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Unity Catalog Volumes DDL Specification
Create Managed and External Volumes within schemas.

In [ ]:
# Volume DDL Syntax:
volume_sql = """
-- Managed Volume: Databricks manages underlying storage path
CREATE VOLUME IF NOT EXISTS enterprise_dev.finance.raw_receipts_managed
COMMENT 'Managed volume storing customer invoice PDFs and scans';

-- External Volume: Governs existing cloud bucket directory
CREATE EXTERNAL VOLUME IF NOT EXISTS enterprise_dev.finance.partner_drops_external
LOCATION 's3://vendor-inbound-bucket/claims/'
COMMENT 'External volume pointing to vendor claims inbox';
"""

print("Volume DDL defined:")
print(volume_sql)

## 6. Interacting with Volumes via POSIX Paths and DBUtils
Read and write files directly using the unified /Volumes path.

In [ ]:
# Databricks supports standard POSIX path access in Python:
# volume_file_path = "/Volumes/enterprise_dev/finance/raw_receipts_managed/invoice_1001.pdf"
# with open(volume_file_path, "rb") as f:
#     pdf_bytes = f.read()

# Using dbutils.fs on Volumes:
# dbutils.fs.ls("/Volumes/enterprise_dev/finance/raw_receipts_managed")

print("Volumes provide direct POSIX file path access for Python libraries (OpenCV, PIL, PyPDF2)!")

## 7. Governing File Access with Unity Catalog Permissions
Grant read and write privileges on Volumes.

In [ ]:
# Access Control Commands for Volumes:
# GRANT READ VOLUME ON VOLUME enterprise_dev.finance.raw_receipts_managed TO `finance_auditors`;
# GRANT WRITE VOLUME ON VOLUME enterprise_dev.finance.raw_receipts_managed TO `data_engineers`;

print("Fine-grained volume access control mapped.")

## 9. Key Architectural Takeaways
- **Unity Catalog Volumes extend data governance to non-tabular files (PDFs, images, CSVs, models).**
- **Volumes provide standard POSIX path syntax (`/Volumes/catalog/schema/volume/file.ext`) compatible with Python libraries.**
- **Volumes deprecate insecure legacy DBFS mounts by bringing unified access controls and audit logs.**

## 10. Senior Data Engineering Interview Questions
1. What is a Unity Catalog Volume and why was it introduced to replace legacy DBFS mounts?
2. What is the difference between a Managed Volume and an External Volume?
3. How do data scientists access files inside a Volume using standard Python `open()`?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
